In [0]:
# Silver ACT Table Pipeline
# Purpose: Clean, type, and consolidate ACT test score data from bronze layer
# Output: workspace.silver.act

from pyspark.sql.functions import col, coalesce, when, regexp_replace, concat_ws, count
from pyspark.sql.types import DoubleType, IntegerType

print("=== ACT Silver Layer Pipeline ===\n")
print("Loading bronze tables...")

# Load source tables
act_recent = spark.table("workspace.bronze.act")
act_historical = spark.table("workspace.bronze.act_highest")

print(f"  Recent: {act_recent.count():,} rows")
print(f"  Historical: {act_historical.count():,} rows")

In [0]:
print("\n=== Transforming to Silver ===\n")
print("Applying transformations:")
print("  • Consolidate district code columns (SCHOOL_DSTRCT_CD + SCHOOL_DISTRCT_CD)")
print("  • Create institution_key (district_code + institution_number)")
print("  • Clean decimal formatting in count columns (.0 suffix)")
print("  • Convert TFS (Too Few Students) to NULL")
print("  • Cast numeric columns to proper types (int/double)")
print("  • Standardize column names to snake_case\n")

act_all = act_recent.unionByName(act_historical, allowMissingColumns=True)
print(f"Total bronze rows: {act_all.count():,}\n")

act_silver = act_all.select(
    col('LONG_SCHOOL_YEAR').alias('school_year'),
    col('INSTN_NUMBER').alias('institution_number'),
    col('INSTN_NAME').alias('institution_name'),
    coalesce(col('SCHOOL_DSTRCT_CD'), col('SCHOOL_DISTRCT_CD')).alias('district_code'),
    col('SCHOOL_DSTRCT_NM').alias('district_name'),
    col('SUBGRP_DESC').alias('subgroup'),
    col('TEST_CMPNT_TYP_CD').alias('test_component'),
    
    # Composite join key for institution
    concat_ws('_', coalesce(col('SCHOOL_DSTRCT_CD'), col('SCHOOL_DISTRCT_CD')), col('INSTN_NUMBER')).alias('institution_key'),
    
    # Counts - strip trailing .0 and cast to int
    when(col('NATIONAL_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('NATIONAL_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('NATIONAL_NUM_TESTED_CNT'), '\\.0*$', '')).cast(IntegerType()).alias('national_num_tested'),
    
    when(col('STATE_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('STATE_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('STATE_NUM_TESTED_CNT'), '\\.0*$', '')).cast(IntegerType()).alias('state_num_tested'),
    
    when(col('DSTRCT_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('DSTRCT_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('DSTRCT_NUM_TESTED_CNT'), '\\.0*$', '')).cast(IntegerType()).alias('district_num_tested'),
    
    when(col('INSTN_NUM_TESTED_CNT') == 'TFS', None)
        .when(col('INSTN_NUM_TESTED_CNT').isNull(), None)
        .otherwise(regexp_replace(col('INSTN_NUM_TESTED_CNT'), '\\.0*$', '')).cast(IntegerType()).alias('institution_num_tested'),
    
    # Scores - handle TFS
    when(col('NATIONAL_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('NATIONAL_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('NATIONAL_AVG_SCORE_VAL')).cast(DoubleType()).alias('national_avg_score'),
    
    when(col('STATE_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('STATE_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('STATE_AVG_SCORE_VAL')).cast(DoubleType()).alias('state_avg_score'),
    
    when(col('DSTRCT_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('DSTRCT_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('DSTRCT_AVG_SCORE_VAL')).cast(DoubleType()).alias('district_avg_score'),
    
    when(col('INSTN_AVG_SCORE_VAL') == 'TFS', None)
        .when(col('INSTN_AVG_SCORE_VAL').isNull(), None)
        .otherwise(col('INSTN_AVG_SCORE_VAL')).cast(DoubleType()).alias('institution_avg_score'),
    
    col('#ASSMT_CD').alias('assessment_code'),
    col('HIGHEST_RECENT_IND').alias('highest_recent_indicator'),
    col('source_year'),
    col('source_file')
)

print("Sample composite scores:")
act_silver.filter("test_component = 'Composite' AND subgroup = 'All Students'").select(
    'school_year', 'institution_name', 'institution_avg_score', 'institution_num_tested'
).orderBy('school_year').show(5, truncate=False)

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max

print("\n=== Data Quality Validation ===\n")

# 1. Row count validation
print("1. Row Count:")
bronze_total = act_recent.count() + act_historical.count()
silver_total = act_silver.count()
print(f"   Bronze: {bronze_total:,} rows")
print(f"   Silver: {silver_total:,} rows")
if bronze_total == silver_total:
    print("   ✓ No data loss")
else:
    print(f"   ⚠️ Row count mismatch: {abs(bronze_total - silver_total):,} rows difference")

# 2. Score range validation
print("\n2. Score Range (ACT valid range: 1-36):")
score_ranges = act_silver.filter(
    "test_component = 'Composite' AND institution_avg_score IS NOT NULL"
).agg(
    spark_min('institution_avg_score').alias('min_score'),
    spark_max('institution_avg_score').alias('max_score')
).collect()[0]

print(f"   Min: {score_ranges['min_score']}")
print(f"   Max: {score_ranges['max_score']}")
if 1 <= score_ranges['min_score'] and score_ranges['max_score'] <= 36:
    print("   ✓ All scores valid")
else:
    print("   ⚠️ Scores outside expected range")

# 3. Institution key validation
print("\n3. Institution Key:")
key_check = act_silver.filter("institution_key IS NOT NULL").count()
print(f"   Rows with institution_key: {key_check:,}")
if key_check == silver_total:
    print("   ✓ All rows have institution_key")
else:
    print(f"   ⚠️ {silver_total - key_check:,} rows missing institution_key")

print("\n✓ Validation complete")

In [0]:
print("\n=== Writing to Silver Layer ===\n")

target_table = "workspace.silver.act"

act_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

final_count = spark.table(target_table).count()
print(f"✓ Table written: {target_table}")
print(f"  Rows: {final_count:,}")
print(f"  Columns: {len(spark.table(target_table).columns)}")

In [0]:
print("\n=== Sample Output ===\n")

print("Recent composite scores (2024-25, All Students):\n")
spark.table("workspace.silver.act").filter(
    "test_component = 'Composite' AND subgroup = 'All Students' AND school_year = '2024-25'"
).select(
    'institution_key',
    'institution_name',
    'district_name',
    'institution_avg_score',
    'institution_num_tested'
).orderBy('institution_name').show(5, truncate=False)

print("\nTable schema (key columns):\n")
spark.sql("DESCRIBE TABLE workspace.silver.act").show(10, truncate=False)

print("\n" + "="*60)
print("✓ Pipeline Complete")
print("="*60)
print(f"\nOutput: workspace.silver.act")
print(f"Purpose: Cleaned ACT test scores with proper types")
print(f"Key Features:")
print(f"  • institution_key: Stable join key (district_code_institution_number)")
print(f"  • Numeric columns properly typed (int/double)")
print(f"  • TFS values converted to NULL")
print(f"  • Ready for analysis and gold layer transformations")